# Challenge Lab 3 - Predicting Airplane Delays

**Educate edition, scaled down.** Replaces
`en_us/Flight_Delay-Student.ipynb`.

## Business scenario

You work for a travel booking website that wants to tell customers, at
booking time, whether a flight to or from a busy US airport is likely
to be delayed.

You have been given US Department of Transportation on-time performance
data. Your task is to build a model that predicts whether a flight will
be delayed.

## Objectives
* Process and create a dataset from downloaded ZIP files
* Perform exploratory data analysis (EDA)
* Establish a baseline model
* Perform hyperparameter optimisation
* Use metrics to compare model performance

## IMPORTANT - how this differs from the original lab

The original AWS Academy lab downloads **72 monthly files covering
2013-2018**, roughly 1.7 GB compressed and far more once expanded. That
requires a 25 GB volume, a large notebook instance, and well over an
hour of compute.

**This version downloads 2 months by default** (about 50 MB). It
covers exactly the same pipeline. Know the trade-off you are making:
your model will be less accurate than one trained on six years of data,
and seasonal effects will be invisible.

Increase `MONTHS` below if you have the budget and the disk space.

In [ ]:
# ================= LAB CONFIGURATION =================
# Each entry is (year, month). Each file is roughly 25 MB compressed.
# 2 months keeps this cheap. Add more if you have budget + disk.
MONTHS = [(2018, 1), (2018, 7)]     # one winter month, one summer month

USE_MANAGED_SAGEMAKER = False       # keep False for the cheap track
SAMPLE_ROWS = 200_000               # cap rows to keep memory/time sane
# =====================================================

import warnings; warnings.simplefilter('ignore')
import os, io, zipfile, requests, json
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid')

print('Will download', len(MONTHS), 'month(s)')

## Step 1 - Download and extract the data

Source: Bureau of Transportation Statistics, "Reporting Carrier On-Time
Performance". Each ZIP contains one CSV for one month.

This takes a few minutes. Watch the disk usage printed at the end.

In [ ]:
BASE = ('https://transtats.bts.gov/PREZIP/'
        'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip')

os.makedirs('flight_data', exist_ok=True)

for y, m in MONTHS:
    url = BASE.format(y=y, m=m)
    target = f'flight_data/{y}_{m}.csv'
    if os.path.exists(target):
        print('Already have', target); continue
    print('Downloading', url)
    try:
        r = requests.get(url, timeout=600)
        r.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(r.content))
        name = [n for n in z.namelist() if n.lower().endswith('.csv')][0]
        with z.open(name) as src, open(target, 'wb') as dst:
            dst.write(src.read())
        print('  ->', target, round(os.path.getsize(target)/1e6, 1), 'MB')
    except Exception as e:
        print('  FAILED:', type(e).__name__, e)

print()
print('Files on disk:')
for f in sorted(os.listdir('flight_data')):
    print(' ', f, round(os.path.getsize('flight_data/'+f)/1e6, 1), 'MB')

## Step 2 - Load only the columns you need

These files have well over 100 columns. Loading all of them wastes
memory and time. Selecting columns up front is the single most
effective optimisation in this lab.

**Critical: avoiding target leakage.** Columns such as `ArrDelay`,
`DepDelay`, `WheelsOff` or `ActualElapsedTime` are only known *after*
the flight has happened. Including them would give a model that scores
beautifully and is completely useless at booking time. We use only
information available **when the customer books**.

In [ ]:
USE_COLS = [
    'Year','Quarter','Month','DayofMonth','DayOfWeek','FlightDate',
    'Reporting_Airline','Origin','Dest','CRSDepTime','CRSArrTime',
    'Distance','DepDel15','Cancelled','Diverted',
]

frames = []
for f in sorted(os.listdir('flight_data')):
    if not f.endswith('.csv'):
        continue
    d = pd.read_csv(f'flight_data/{f}', usecols=lambda c: c in USE_COLS,
                    low_memory=False)
    frames.append(d)
    print(f, d.shape)

df = pd.concat(frames, ignore_index=True)
print()
print('Combined:', df.shape)
df.head()

## Step 3 - Define the target and clean the data

`DepDel15` is the DOT's own flag: 1 if the flight departed 15+ minutes
late, 0 otherwise. That is our target.

Cancelled and diverted flights are removed - they are a different
prediction problem, and their delay fields are unreliable.

In [ ]:
before = len(df)
df = df[(df['Cancelled'] == 0) & (df['Diverted'] == 0)]
df = df.dropna(subset=['DepDel15'])
df['target'] = df['DepDel15'].astype(int)

print(f'Removed {before - len(df):,} cancelled/diverted/missing rows')
print(f'Remaining: {len(df):,}')
print()
print('Target balance:')
print(df['target'].value_counts())
print((df['target'].value_counts(normalize=True) * 100).round(1))
print()
print(f'Majority-class baseline accuracy: {max(df["target"].mean(), 1-df["target"].mean()):.1%}')

### Focus on the busiest airports

The business requirement is the **busiest airports for domestic
travel**. Narrowing to those makes the problem both more relevant and
computationally lighter.

In [ ]:
top_airports = df['Origin'].value_counts().nlargest(15).index.tolist()
print('Top 15 origin airports:', top_airports)

df = df[df['Origin'].isin(top_airports) & df['Dest'].isin(top_airports)]
print(f'\nRows after filtering to top-15 to top-15: {len(df):,}')

if len(df) > SAMPLE_ROWS:
    df = df.sample(SAMPLE_ROWS, random_state=42)
    print(f'Sampled down to {len(df):,} rows')

## Step 4 - Exploratory data analysis

Before modelling, look for signal. If a feature shows no relationship
with the target here, it is unlikely to help the model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# delay rate by hour of scheduled departure
df['dep_hour'] = (df['CRSDepTime'] // 100).clip(0, 23)
df.groupby('dep_hour')['target'].mean().plot(
    ax=axes[0,0], marker='o', title='Delay rate by scheduled departure hour')
axes[0,0].set_ylabel('P(delayed)')

# by day of week
df.groupby('DayOfWeek')['target'].mean().plot(
    kind='bar', ax=axes[0,1], rot=0, title='Delay rate by day of week')

# by month
df.groupby('Month')['target'].mean().plot(
    kind='bar', ax=axes[1,0], rot=0, title='Delay rate by month')

# by airline
df.groupby('Reporting_Airline')['target'].mean().sort_values().plot(
    kind='barh', ax=axes[1,1], title='Delay rate by airline')

plt.tight_layout(); plt.show()

The departure-hour chart is usually the most striking: delay rates
climb steadily through the day as disruptions cascade. Early-morning
flights are far more reliable. That single feature carries a lot of the
model's predictive power.

In [ ]:
print('Delay rate by origin airport:')
print((df.groupby('Origin')['target'].agg(['mean','size'])
         .sort_values('mean', ascending=False)
         .rename(columns={'mean':'delay_rate','size':'flights'})
         .assign(delay_rate=lambda x: (x.delay_rate*100).round(1))))

## Step 5 - Feature engineering and encoding

`Origin`, `Dest` and `Reporting_Airline` are nominal categoricals -
exactly the situation from Lab 3.3. We one-hot encode them.

We also derive a few features the raw data does not provide directly.

In [ ]:
df['dep_hour']  = (df['CRSDepTime'] // 100).clip(0, 23)
df['arr_hour']  = (df['CRSArrTime'] // 100).clip(0, 23)
df['is_weekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)
df['route'] = df['Origin'] + '_' + df['Dest']

numeric = ['Month','DayofMonth','DayOfWeek','Distance',
           'dep_hour','arr_hour','is_weekend']
nominal = ['Reporting_Airline','Origin','Dest']

X = pd.concat([
    df[numeric].reset_index(drop=True),
    pd.get_dummies(df[nominal], drop_first=True).astype(int).reset_index(drop=True),
], axis=1)
y = df['target'].reset_index(drop=True)

print('Feature matrix:', X.shape)
X.head()

## Step 6 - Train / validation / test split

Same stratified approach as Lab 3.4.

In [ ]:
from sklearn.model_selection import train_test_split

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, random_state=42, stratify=y_tv)

for n, yy in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f'{n:11s} {len(yy):7,} rows   delay rate {yy.mean():.1%}')

## Step 7 - Establish a baseline model

Always start with a trivial baseline. If your sophisticated model
cannot beat "predict the majority class every time", something is
wrong.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

majority = int(y_train.mode()[0])
naive_pred = np.full(len(y_test), majority)

print('=== Naive baseline: always predict the majority class ===')
print(f'Accuracy: {accuracy_score(y_test, naive_pred):.1%}')
print(f'Recall  : {recall_score(y_test, naive_pred, zero_division=0):.1%}'
      '   <- it never catches a single delay')
print()
print('This is why accuracy alone is a trap on imbalanced data.')

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42,
}

baseline_model = xgb.train(
    params, dtrain, num_boost_round=200,
    evals=[(dtrain,'train'), (dval,'validation')],
    early_stopping_rounds=20, verbose_eval=25)

p_base = baseline_model.predict(dtest)
pred_base = (p_base > 0.5).astype(int)

baseline_metrics = {
    'accuracy':  accuracy_score(y_test, pred_base),
    'precision': precision_score(y_test, pred_base, zero_division=0),
    'recall':    recall_score(y_test, pred_base, zero_division=0),
    'f1':        f1_score(y_test, pred_base, zero_division=0),
    'auc':       roc_auc_score(y_test, p_base),
}
print()
print('=== XGBoost baseline ===')
for k, v in baseline_metrics.items():
    print(f'{k:10s} {v:.4f}')

### The class-imbalance problem, made visible

Look at the recall above. Because only ~20% of flights are delayed, the
model at a 0.5 threshold plays it safe and predicts "on time" for
almost everything. High accuracy, near-useless product.

`scale_pos_weight` tells XGBoost to weight the minority class more
heavily. This is the key fix.

In [ ]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {spw:.2f}')

params_bal = dict(params, scale_pos_weight=spw)
balanced_model = xgb.train(
    params_bal, dtrain, num_boost_round=200,
    evals=[(dval,'validation')],
    early_stopping_rounds=20, verbose_eval=False)

p_bal = balanced_model.predict(dtest)
pred_bal = (p_bal > 0.5).astype(int)

balanced_metrics = {
    'accuracy':  accuracy_score(y_test, pred_bal),
    'precision': precision_score(y_test, pred_bal, zero_division=0),
    'recall':    recall_score(y_test, pred_bal, zero_division=0),
    'f1':        f1_score(y_test, pred_bal, zero_division=0),
    'auc':       roc_auc_score(y_test, p_bal),
}
print()
print(pd.DataFrame({'baseline': baseline_metrics,
                    'class-weighted': balanced_metrics}).round(4))
print()
print('Accuracy went DOWN, recall went UP. For this product - warning')
print('customers about delays - recall is what the business cares about.')

## Step 8 - Hyperparameter optimisation

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import scipy.stats as st

param_dist = {
    'max_depth':        [4, 6, 8, 10],
    'learning_rate':    st.uniform(0.02, 0.2),
    'n_estimators':     [100, 200, 300],
    'min_child_weight': [1, 5, 10],
    'subsample':        st.uniform(0.6, 0.4),
    'colsample_bytree': st.uniform(0.6, 0.4),
}

search = RandomizedSearchCV(
    XGBClassifier(objective='binary:logistic', eval_metric='auc',
                  scale_pos_weight=spw, random_state=42),
    param_distributions=param_dist,
    n_iter=12,             # keep small - each fit is a full training run
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print()
print(f'Best CV AUC: {search.best_score_:.4f}')
print('Best params:')
for k, v in sorted(search.best_params_.items()):
    print(f'  {k:18s} {v}')

In [ ]:
tuned = search.best_estimator_
p_tuned = tuned.predict_proba(X_test)[:, 1]
pred_tuned = (p_tuned > 0.5).astype(int)

tuned_metrics = {
    'accuracy':  accuracy_score(y_test, pred_tuned),
    'precision': precision_score(y_test, pred_tuned, zero_division=0),
    'recall':    recall_score(y_test, pred_tuned, zero_division=0),
    'f1':        f1_score(y_test, pred_tuned, zero_division=0),
    'auc':       roc_auc_score(y_test, p_tuned),
}

comparison = pd.DataFrame({
    'naive':          {'accuracy': accuracy_score(y_test, naive_pred),
                       'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': 0.5},
    'xgb baseline':   baseline_metrics,
    'class-weighted': balanced_metrics,
    'tuned':          tuned_metrics,
}).round(4)

display(comparison)

In [ ]:
ax = comparison.T[['accuracy','recall','f1','auc']].plot(
    kind='bar', figsize=(11, 5), rot=0,
    title='Model comparison - flight delay prediction (test set)')
ax.set_ylabel('score'); ax.set_ylim(0, 1.0)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## Step 9 - What the model learned

In [ ]:
imp = pd.Series(tuned.feature_importances_, index=X.columns)
imp = imp.sort_values(ascending=False).head(20)

imp.sort_values().plot(kind='barh', figsize=(9, 7),
                       title='Top 20 features by importance')
plt.tight_layout(); plt.show()
print(imp.round(4))

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_curve

plt.figure(figsize=(6, 5))
for name, p in [('baseline', p_base), ('class-weighted', p_bal),
                ('tuned', p_tuned)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc_score(y_test,p):.3f})')
plt.plot([0,1],[0,1],'k--',lw=1,label='random')
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC comparison'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## Step 10 - Interpreting the result honestly

A realistic AUC here is roughly **0.74-0.77** (a verified run on two
months of 2018 data gave 0.762). That is a real signal, well above
random - but it is not a solved problem, and you should say so.

Look hard at the comparison table. The plain XGBoost baseline scored
the *highest accuracy* of any model (about 82%) while catching only
about **14%** of actual delays. It achieved that score mostly by
predicting "on time" almost every time. The class-weighted model scores
*lower* accuracy (about 70%) but catches about **67%** of delays. For a
product whose entire purpose is warning customers about delays, the
second model is far more useful and the first is close to worthless.

This is the clearest demonstration in the whole module of why you must
choose your metric to match the business goal.

**Why the ceiling is low:** the biggest single cause of departure
delays is weather, and this dataset contains **no weather data at
all**. We are predicting a weather-driven outcome without observing the
weather. Schedule, airport and airline are proxies at best.

**What would improve it:**
* Join historical weather observations by airport and hour
* Add the delay status of the aircraft's *previous* flight that day -
  delays cascade through aircraft rotations, and this is usually the
  strongest available predictor
* Add airport congestion (scheduled departures in the same hour)
* Train on the full 6 years to capture seasonal and yearly patterns

**Would you ship this?** At AUC ~0.70, a "likely delayed" warning would
be wrong often enough to annoy customers. The honest recommendation is
to gather weather and aircraft-rotation data before productionising.
Recognising that is a better answer than reporting a number.

## Cleanup

This lab downloaded data to `flight_data/`. Free the disk space if you
plan to keep the notebook instance.

In [ ]:
import shutil
size = sum(os.path.getsize('flight_data/'+f) for f in os.listdir('flight_data'))
print(f'flight_data/ is using {size/1e6:.0f} MB')
print('Uncomment the next line to delete it:')
# shutil.rmtree('flight_data')

## Conclusion

You have:
* Processed and created a dataset from downloaded ZIP files
* Performed exploratory data analysis
* Established a baseline model, and beaten a naive one
* Handled class imbalance with `scale_pos_weight`
* Performed hyperparameter optimisation
* Compared models using metrics appropriate to an imbalanced problem
* Assessed, honestly, whether the model is good enough to ship

**Final cleanup: Stop or Delete the notebook instance in the SageMaker
console.**